# Простая нейросеть для категоризации платежей типа 'expense'

В этом ноутбуке реализуем простую нейросеть, которая будет определять категорию расходов на основе текстового описания платежа.

In [ ]:
# Импортируем необходимые библиотеки
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D, GlobalMaxPooling1D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Подготовка тестовых данных

Создадим примеры транзакций для обучения модели

In [ ]:
# Создаем пример данных для обучения
data = {
    'description': [
        'оплата в супермаркете ашан', 'перекресток покупка продуктов', 'лента продукты', 'магнит хлеб молоко',
        'кафе завтрак', 'ресторан ужин', 'столовая обед', 'доставка еды суши',
        'яндекс такси', 'uber поездка', 'метро', 'автобус проезд',
        'оплата жкх', 'квартплата', 'электричество счет', 'вода газ',
        'кинотеатр билеты', 'концерт', 'театр опера', 'музей выставка',
        'аптека лекарства', 'стоматология', 'анализы', 'врач прием',
        'зара одежда', 'обувь', 'спортмастер кроссовки', 'h&m футболка'
    ],
    'category': [
        'продукты', 'продукты', 'продукты', 'продукты',
        'кафе_рестораны', 'кафе_рестораны', 'кафе_рестораны', 'кафе_рестораны',
        'транспорт', 'транспорт', 'транспорт', 'транспорт',
        'жкх', 'жкх', 'жкх', 'жкх',
        'развлечения', 'развлечения', 'развлечения', 'развлечения',
        'здоровье', 'здоровье', 'здоровье', 'здоровье',
        'одежда', 'одежда', 'одежда', 'одежда'
    ]
}

# Создаем DataFrame
df = pd.DataFrame(data)
df.head()

## Подготовка данных для модели

In [ ]:
# Кодируем категории
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['category'])

# Параметры для токенизации текста
max_features = 1000  # максимальное количество слов в словаре
max_len = 20  # максимальная длина последовательности

# Создаем токенайзер и преобразуем тексты в последовательности
tokenizer = Tokenizer(num_words=max_features, lower=True, split=' ')
tokenizer.fit_on_texts(df['description'])
X = tokenizer.texts_to_sequences(df['description'])
X = pad_sequences(X, maxlen=max_len)

# Разделяем данные на обучающий и тестовый наборы
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Выводим количество классов
num_classes = len(label_encoder.classes_)
print(f"Количество классов: {num_classes}")
print(f"Классы: {label_encoder.classes_}")

## Создание модели нейросети

In [ ]:
# Создаем модель
embedding_dim = 128  # размерность векторного представления слов

model = Sequential()
model.add(Embedding(max_features, embedding_dim, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Выводим информацию о модели
model.summary()

## Обучение модели

In [ ]:
# Обучаем модель
epochs = 100
batch_size = 4

history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    verbose=1
)

## Оценка модели

In [ ]:
# Визуализируем процесс обучения
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Точность')
plt.ylabel('Точность')
plt.xlabel('Эпоха')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Потери')
plt.ylabel('Потери')
plt.xlabel('Эпоха')
plt.legend()

plt.tight_layout()
plt.show()

# Оцениваем модель на тестовых данных
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Точность на тестовых данных: {accuracy*100:.2f}%")

## Функция для предсказания категории

In [ ]:
# Функция для предсказания категории нового платежа
def predict_category(description):
    # Преобразуем текст в последовательность
    seq = tokenizer.texts_to_sequences([description])
    padded = pad_sequences(seq, maxlen=max_len)

    # Получаем предсказание
    pred = model.predict(padded)[0]
    predicted_class = np.argmax(pred)
    confidence = pred[predicted_class]

    # Преобразуем числовой класс обратно в категорию
    category = label_encoder.inverse_transform([predicted_class])[0]

    return category, confidence

## Тестирование модели на новых данных

In [ ]:
# Тестируем на новых примерах
test_examples = [
    'оплата в пятерочке',
    'ресторан макдональдс',
    'проезд в автобусе',
    'счет за интернет',
    'покупка билетов в кино',
    'аптека витамины',
    'бершка джинсы',
    'ипотека'
]

for example in test_examples:
    category, confidence = predict_category(example)
    print(f"Описание: '{example}'")
    print(f"Предсказанная категория: '{category}' (уверенность: {confidence*100:.2f}%)\n")

## Сохранение модели для дальнейшего использования

In [ ]:
# Сохраняем модель
model.save('payment_category_model.h5')

# Сохраняем токенайзер и кодировщик меток
import pickle

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('label_encoder.pickle', 'wb') as handle:
    pickle.dump(label_encoder, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Модель и вспомогательные объекты сохранены!")

## Загрузка модели и использование

In [ ]:
# Пример загрузки модели
def load_model_and_dependencies():
    from tensorflow.keras.models import load_model
    import pickle

    # Загружаем модель
    loaded_model = load_model('payment_category_model.h5')

    # Загружаем токенайзер
    with open('tokenizer.pickle', 'rb') as handle:
        loaded_tokenizer = pickle.load(handle)

    # Загружаем энкодер меток
    with open('label_encoder.pickle', 'rb') as handle:
        loaded_label_encoder = pickle.load(handle)

    return loaded_model, loaded_tokenizer, loaded_label_encoder

# Функция для предсказания с загруженной моделью
def predict_with_loaded_model(description, model, tokenizer, label_encoder):
    max_len = 20  # должно совпадать с тем, что использовалось при обучении

    # Преобразуем текст в последовательность
    seq = tokenizer.texts_to_sequences([description])
    padded = pad_sequences(seq, maxlen=max_len)

    # Получаем предсказание
    pred = model.predict(padded)[0]
    predicted_class = np.argmax(pred)
    confidence = pred[predicted_class]

    # Преобразуем числовой класс обратно в категорию
    category = label_encoder.inverse_transform([predicted_class])[0]

    return category, confidence

# Пример использования (закомментировано, чтобы избежать ошибок, если файлы еще не существуют)
# model, tokenizer, label_encoder = load_model_and_dependencies()
# new_payment = "оплата в икеа мебель"
# category, confidence = predict_with_loaded_model(new_payment, model, tokenizer, label_encoder)
# print(f"Платеж: '{new_payment}'")
# print(f"Категория: '{category}' (уверенность: {confidence*100:.2f}%)")